# Session 7 · Building AI-Powered Apps: Local LLMs, Summariser, Chatbot and RAG

Generative AI › Track 4 · AI Architecture and Agents

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-07_local_llm_client.py`

In [1]:
"""
Session 7 · Example 1 — Talking to a local LLM through Ollama's REST API, and sizing models for your laptop.

Ollama serves open models (Qwen 2.5, Llama 3, Mistral, DeepSeek-R1) at localhost:11434.
We size models for your RAM, build the /api/chat body and call it if Ollama is running.

Setup (your own machine):  install Ollama from ollama.com, then
    ollama pull qwen2.5:7b      # or qwen2.5:1.5b on a small laptop
Run:  python session-07_local_llm_client.py
"""
import json
import urllib.request

OLLAMA = "http://localhost:11434"

# 1) Memory estimate: parameters x bytes per weight x ~1.2 overhead (KV cache, runtime)
MODELS = {"qwen2.5:1.5b": 1.5, "mistral:7b": 7.2, "qwen2.5:7b": 7.6, "llama3:8b": 8.0,
          "qwen2.5:14b": 14.7, "qwen2.5:72b": 72.7}
BITS = {"FP16": 16, "Q8": 8, "Q4": 4.5}       # Q4 formats store ~4.5 bits/weight incl. scales
print(f"{'model':14s} {'params':>7s}  " + "  ".join(f"{k:>6s}" for k in BITS) + "   (GB of RAM/VRAM)")
for name, b in MODELS.items():
    gbs = [b * 1e9 * bits / 8 * 1.2 / 1e9 for bits in BITS.values()]
    print(f"{name:14s} {b:6.1f}B  " + "  ".join(f"{g:6.1f}" for g in gbs))

# 2) The request body for a chat call
body = {
    "model": "qwen2.5:7b",
    "messages": [
        {"role": "system", "content": "You are a concise teaching assistant. Answer in 2 sentences."},
        {"role": "user", "content": "What does a transformer's attention layer do?"},
    ],
    "stream": False,
    "options": {"temperature": 0.2, "num_ctx": 4096},
}
print("\nPOST /api/chat body:\n{\n" + ",\n".join(f'  "{k}": {json.dumps(v)}' for k, v in body.items()) + "\n}")


def chat(payload, timeout=60):
    req = urllib.request.Request(OLLAMA + "/api/chat", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())["message"]["content"]


def ollama_running():
    try:
        urllib.request.urlopen(OLLAMA + "/api/tags", timeout=1)
        return True
    except Exception:
        return False

# 3) Call it if available
if ollama_running():
    print("\nModel reply:\n" + chat(body))
else:
    print("\nOllama is not running on this machine, so no reply was generated.")
    print("Start it with:  ollama serve   then   ollama pull qwen2.5:7b   and run this script again.")

model           params    FP16      Q8      Q4   (GB of RAM/VRAM)
qwen2.5:1.5b      1.5B     3.6     1.8     1.0
mistral:7b        7.2B    17.3     8.6     4.9
qwen2.5:7b        7.6B    18.2     9.1     5.1
llama3:8b         8.0B    19.2     9.6     5.4
qwen2.5:14b      14.7B    35.3    17.6     9.9
qwen2.5:72b      72.7B   174.5    87.2    49.1

POST /api/chat body:
{
  "model": "qwen2.5:7b",
  "messages": [{"role": "system", "content": "You are a concise teaching assistant. Answer in 2 sentences."}, {"role": "user", "content": "What does a transformer's attention layer do?"}],
  "stream": false,
  "options": {"temperature": 0.2, "num_ctx": 4096}
}

Ollama is not running on this machine, so no reply was generated.
Start it with:  ollama serve   then   ollama pull qwen2.5:7b   and run this script again.


### Example · `session-07_summarizer.py`

In [2]:
"""
Session 7 · Example 2 — A text summariser: offline extractive engine + LLM (DeepSeek-R1 / Qwen) prompt via Ollama.

Extractive (TF-IDF sentence scores: offline, never invents facts) and abstractive
(LLM via Ollama: fluent, must be checked). DeepSeek-R1 wraps its reasoning in <think> tags.

Run:  python session-07_summarizer.py
"""
import json, math, re, urllib.request
from collections import Counter

ARTICLE = """Karachi's public transport is changing quickly. The Green Line bus rapid transit
opened at the end of 2021 and now carries large numbers of passengers every day.
Commuters say travel times on the corridor have dropped sharply compared with minibuses.
However, feeder routes are still limited, so many riders walk long distances to reach stations.
The city has also introduced electric buses on several routes to reduce noise and air pollution.
Officials plan to integrate ticketing across lines with a single smart card.
Critics argue that maintenance funding must be secured, or service quality will decline.
Experts recommend publishing ridership data so that planners can match buses to demand.""".replace("\n", " ")

STOP = set("the a an and or of to in on is are was were has have with so that or for by at as be will must can many now still also which this its".split())


def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def words(s):
    return [w for w in re.findall(r"[a-z]+", s.lower()) if w not in STOP]


def extractive_summary(text, k=3):
    sents = sentences(text)
    docs = [Counter(words(s)) for s in sents]
    df = Counter(w for d in docs for w in d)
    n = len(sents)
    scores = []
    for i, d in enumerate(docs):
        tfidf = sum(c * math.log(n / df[w]) for w, c in d.items())
        scores.append((tfidf / (1 + sum(d.values())) ** 0.5, i))   # mild length normalisation
    keep = sorted(i for _, i in sorted(scores, reverse=True)[:k])  # keep original order
    return [sents[i] for i in keep], {i: round(s, 2) for s, i in scores}


def strip_thinking(reply):
    return re.sub(r"<think>.*?</think>", "", reply, flags=re.S).strip()


summary, scores = extractive_summary(ARTICLE, k=3)
print(f"Article: {len(sentences(ARTICLE))} sentences, {len(ARTICLE.split())} words")
print("\nExtractive summary (3 sentences):")
for s in summary:
    print(" -", s)
print(f"Compression: {sum(len(s.split()) for s in summary)} / {len(ARTICLE.split())} words")

prompt = ("Summarise the article in exactly 3 bullet points of at most 15 words each. "
          "Use only facts from the article; do not add numbers that are not in it.\n\n"
          f'Article: """{ARTICLE}"""')
body = {"model": "deepseek-r1:7b", "prompt": prompt, "stream": False, "options": {"temperature": 0.1}}

print("\nstrip_thinking demo:", repr(strip_thinking("<think>3 bullets: Green Line, feeders...</think>\n- Green Line cut travel times.")))

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
    req = urllib.request.Request("http://localhost:11434/api/generate", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    reply = json.loads(urllib.request.urlopen(req, timeout=120).read())["response"]
    print("\nLLM summary:\n" + strip_thinking(reply))
except Exception:
    print("\n(Ollama not running here: the offline extractive summary above is the result.)")

Article: 8 sentences, 109 words

Extractive summary (3 sentences):
 - The Green Line bus rapid transit opened at the end of 2021 and now carries large numbers of passengers every day.
 - However, feeder routes are still limited, so many riders walk long distances to reach stations.
 - Officials plan to integrate ticketing across lines with a single smart card.
Compression: 48 / 109 words

strip_thinking demo: '- Green Line cut travel times.'

(Ollama not running here: the offline extractive summary above is the result.)


### Example · `session-07_rag_search.py`

In [3]:
"""
Session 7 · Example 3 — Retrieval-Augmented Generation (RAG): retrieve, then build a grounded prompt.

Retrieve the most similar handbook chunks, then build a prompt that answers ONLY from
them with citations. Real systems use neural embeddings + a vector DB; TF-IDF keeps it offline.

Run:  python session-07_rag_search.py
Creates: session-07_rag_scores.png
"""
import math, re
from collections import Counter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DOCS = {
    "attendance.md": "Students must attend at least 75 percent of classes in each course. Below 75 percent the student is not allowed to sit the final exam.",
    "exams.md": "Final exams are held in the last two weeks of the semester. A student who misses an exam for medical reasons may apply for a deferred exam within 7 days with a medical certificate.",
    "fees.md": "Semester fees are due before the first day of classes. A late fee of 5,000 rupees applies after the due date. Fees can be paid in two instalments with approval.",
    "library.md": "The library is open from 8 am to 9 pm on weekdays. Students may borrow up to 4 books for 14 days. The fine for a late book is 50 rupees per day.",
    "wifi.md": "Campus wifi uses your student ID and portal password. If you cannot connect, reset your password on the portal or visit the IT help desk in Block B.",
    "hostel.md": "Hostel rooms are allocated by merit and distance from home. Guests are allowed in the common room until 8 pm. The hostel deposit is refunded within 30 days of leaving.",
}
STOP = set("the a an and or of to in on is are be for by at as it if you your may must can from with within each after per up".split())


# Keyword vectors miss synonyms ("sick" vs "medical"). Neural embeddings learn these
# links automatically; here a small synonym table plays that role.
SYNONYMS = {"sick": "medical", "ill": "medical", "missed": "misses", "pay": "paid", "books": "book",
            "take": "borrow", "internet": "wifi", "parts": "instalments"}


def tokens(text, expand=False):
    ws = [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]
    return ws + [SYNONYMS[w] for w in ws if expand and w in SYNONYMS]


df = Counter(w for d in DOCS.values() for w in set(tokens(d)))
N = len(DOCS)


def vec(text, expand=False):
    tf = Counter(tokens(text, expand))
    return {w: c * (math.log((N + 1) / (df.get(w, 0) + 1)) + 1) for w, c in tf.items()}


def cosine(a, b):
    dot = sum(a[w] * b.get(w, 0) for w in a)
    norm = math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values()))
    return dot / norm if norm else 0.0


INDEX = {name: vec(text) for name, text in DOCS.items()}


def retrieve(question, k=2, min_score=0.10, expand=True):
    q = vec(question, expand)
    ranked = sorted(((cosine(q, v), name) for name, v in INDEX.items()), reverse=True)
    return ranked, [(s, n) for s, n in ranked[:k] if s >= min_score]


def grounded_prompt(question, hits):
    context = "\n".join(f"[{n}] {DOCS[n]}" for _, n in hits) or "(no relevant documents found)"
    return ("Answer the question using ONLY the context. Cite sources in [brackets]. "
            "If the context does not contain the answer, say you don't know.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\nAnswer:")


if __name__ == "__main__":      # lets session-07_project_chatbot.py import the retriever
    questions = ["I missed my exam because I was sick, what can I do?",
                 "How much is the fine if I return a library book late?",
                 "Who won the cricket match yesterday?"]
    plain = retrieve(questions[0], expand=False)[0][0]
    print(f"Keyword-only best match for question 1: {plain[1]} ({plain[0]:.2f}) -> below 0.10, nothing retrieved")
    print("With synonym expansion (what embeddings do for you):\n")
    all_ranked = {}
    for q in questions:
        ranked, hits = retrieve(q)
        all_ranked[q] = ranked
        print(f"Q: {q}")
        print("   top matches:", ", ".join(f"{n} ({s:.2f})" for s, n in ranked[:3]))
        print("   used as context:", [n for _, n in hits] or "none -> the model must say it does not know")
    print("\nGrounded prompt for question 2:\n" + grounded_prompt(questions[1], retrieve(questions[1])[1]))

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
    for ax, q in zip(axes, questions):
        ranked = sorted(all_ranked[q], key=lambda t: t[1])                     # alphabetical rows
        ax.barh([n for _, n in ranked], [sc for sc, _ in ranked], color=["#FF6B1A" if sc >= 0.10 else "#A78BFA" for sc, _ in ranked])
        ax.axvline(0.10, color="#F43F5E", ls="--", lw=1); ax.set_title(q[:34] + "...", fontsize=9); ax.set_xlim(0, 0.8)
    axes[1].set_xlabel("Cosine similarity (dashed line = minimum score)")
    fig.tight_layout(); fig.savefig("session-07_rag_scores.png", dpi=110)
    print("\nSaved chart: session-07_rag_scores.png")

Keyword-only best match for question 1: exams.md (0.06) -> below 0.10, nothing retrieved
With synonym expansion (what embeddings do for you):

Q: I missed my exam because I was sick, what can I do?
   top matches: exams.md (0.17), attendance.md (0.03), wifi.md (0.00)
   used as context: ['exams.md']
Q: How much is the fine if I return a library book late?
   top matches: library.md (0.28), fees.md (0.04), wifi.md (0.00)
   used as context: ['library.md']
Q: Who won the cricket match yesterday?
   top matches: exams.md (0.08), wifi.md (0.00), library.md (0.00)
   used as context: none -> the model must say it does not know

Grounded prompt for question 2:
Answer the question using ONLY the context. Cite sources in [brackets]. If the context does not contain the answer, say you don't know.

Context:
[library.md] The library is open from 8 am to 9 pm on weekdays. Students may borrow up to 4 books for 14 days. The fine for a late book is 50 rupees per day.

Question: How much is the fine i

### Example · `session-07_project_chatbot.py`

In [4]:
"""
Session 7 · Mini project (full solution) — "CampusBot": a RAG chatbot on Qwen 2.5 via Ollama, with an offline fallback.

Real-life problem: student affairs wants a chatbot that answers ONLY from the handbook
with citations, remembers the conversation for follow-ups, declines out-of-scope questions
with a contact, and runs on a local Qwen 2.5 model via Ollama (offline fallback if not running).
It reuses the handbook and retriever from session-07_rag_search.py (keep both files together).

Run:  python session-07_project_chatbot.py                # scripted demo
      python session-07_project_chatbot.py --interactive  # chat yourself (type 'quit')
"""
import importlib.util, json, re, sys, urllib.request
from collections import Counter, deque

try:
    spec = importlib.util.spec_from_file_location("rag", "session-07_rag_search.py")
    rag = importlib.util.module_from_spec(spec); spec.loader.exec_module(rag)   # DOCS, tokens, retrieve
except FileNotFoundError:            # in the notebook/Colab the RAG cell above already defined them
    import types; rag = types.SimpleNamespace(DOCS=DOCS, tokens=tokens, retrieve=retrieve)

CONTACT = "student.affairs@campus.example (Room A-12, 9 am to 4 pm)"
SYSTEM = ("You are CampusBot, a polite assistant for students. Answer in at most 2 sentences using ONLY "
          "the handbook context, and end with the source file in [brackets]. If the context does not "
          f"answer the question, say you don't know and suggest contacting {CONTACT}.")
MODEL = "qwen2.5:7b"


class CampusBot:
    def __init__(self, memory_turns=3):
        self.history = deque(maxlen=memory_turns * 2)    # sliding window of user/assistant messages
        self.topic = None
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1); self.online = True
        except Exception:
            self.online = False

    def retrieve(self, question):
        ranked, hits = rag.retrieve(question, k=1)
        score = ranked[0][0]
        # A follow-up ("and how many...?", "what about it?") has no topic of its own: reuse the remembered one.
        follow_up = re.match(r"\s*(and|also|what about|how about)\b", question.lower()) or \
            re.search(r"\b(it|that|them|those)\b", question.lower())
        if not hits and self.topic and follow_up:
            return self.topic, score, True
        return (hits[0][1] if hits else None), score, False

    def _llm(self, question, doc):
        context = f"[{doc}] {rag.DOCS[doc]}" if doc else "(no relevant section)"
        messages = [{"role": "system", "content": SYSTEM}, *self.history,
                    {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]
        body = json.dumps({"model": MODEL, "messages": messages, "stream": False, "options": {"temperature": 0.1}})
        req = urllib.request.Request("http://localhost:11434/api/chat", data=body.encode(),
                                     headers={"Content-Type": "application/json"})
        return json.loads(urllib.request.urlopen(req, timeout=120).read())["message"]["content"]

    def _fallback(self, question, doc):
        if doc is None:
            return f"I'm sorry, I don't know that from the handbook. Please contact {CONTACT}."
        q = set(rag.tokens(question, expand=True))
        sents = re.split(r"(?<=\.)\s+", rag.DOCS[doc])
        return max(sents, key=lambda s: len(q & set(rag.tokens(s)))) + f" [{doc}]"   # best-overlap sentence

    def ask(self, question):
        doc, score, from_memory = self.retrieve(question)
        answer = self._llm(question, doc) if self.online else self._fallback(question, doc)
        self.topic = doc or self.topic
        self.history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": answer}])
        return answer, doc, score, from_memory


if __name__ == "__main__":
    bot = CampusBot()
    print(f"CampusBot ready | engine: {'Ollama ' + MODEL if bot.online else 'offline extractive fallback'}\n")
    if "--interactive" in sys.argv:
        while (q := input("You: ").strip()).lower() not in {"quit", "exit"}:
            print("Bot:", bot.ask(q)[0], "\n")
        sys.exit()
    stats = Counter()
    for q in ["When are final exams held?", "I was sick and missed one, what now?",
              "How much is the late fine for a library book?", "And how many can I take at once?",
              "Can I pay my fees in parts?", "What is the cafeteria menu today?"]:
        answer, doc, score, mem = bot.ask(q)
        stats["answered" if doc else "declined"] += 1
        tag = f"{doc} via memory" if mem else (f"{doc}, score {score:.2f}" if doc else f"no match, best {score:.2f}")
        print(f"You: {q}\nBot: {answer}\n     ({tag})\n")
    print("Session stats:", dict(stats), f"| memory window holds {len(bot.history)} messages")

CampusBot ready | engine: offline extractive fallback

You: When are final exams held?
Bot: Final exams are held in the last two weeks of the semester. [exams.md]
     (exams.md, score 0.28)

You: I was sick and missed one, what now?
Bot: A student who misses an exam for medical reasons may apply for a deferred exam within 7 days with a medical certificate. [exams.md]
     (exams.md, score 0.18)

You: How much is the late fine for a library book?
Bot: The fine for a late book is 50 rupees per day. [library.md]
     (library.md, score 0.34)

You: And how many can I take at once?
Bot: Students may borrow up to 4 books for 14 days. [library.md]
     (library.md via memory)

You: Can I pay my fees in parts?
Bot: Fees can be paid in two instalments with approval. [fees.md]
     (fees.md, score 0.27)

You: What is the cafeteria menu today?
Bot: I'm sorry, I don't know that from the handbook. Please contact student.affairs@campus.example (Room A-12, 9 am to 4 pm).
     (no match, best 0.00)

